In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

import types
dfthcsos = types.ModuleType('dfthcsos')
sys.modules['dfthcsos'] = dfthcsos
scoreinfra = types.ModuleType('dfthcsos.scoreinfra')
sys.modules['dfthcsos.scoreinfra'] = scoreinfra
dfthcsos.scoreinfra = scoreinfra
scoreinfra.__path__ = []
import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith('__'):
        setattr(scoreinfra, attr, getattr(dm, attr))
from verifier import circuits as vc
circuits_mod = types.ModuleType('dfthcsos.scoreinfra.circuits')
for attr in dir(vc):
    if not attr.startswith('__'):
        setattr(circuits_mod, attr, getattr(vc, attr))
sys.modules['dfthcsos.scoreinfra.circuits'] = circuits_mod
scoreinfra.circuits = circuits_mod

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))

# Use Michael's factorization (has brot, words1, words2)
fact_path = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/fact/dfthc_r10_b129_c46_brot8.npz')
shutil.copy2(fact_path, sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / 'fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers'))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'fin.npz').exists())


In [ ]:
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / 'fin.npz'), str(RUNS / 'g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers'))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'g.npz').exists())


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('dfthcsos_verify', str(COMP / 'verifiers' / 'dfthcsos' / 'verify.py'))
dfthcsos_verify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dfthcsos_verify)
CV = dfthcsos_verify.verify

final_npz = RUNS / 'g.npz'
if not final_npz.exists():
    final_npz = RUNS / 'fin.npz'

print('Using factorization:', final_npz)
shutil.copy2(final_npz, sub_dir / 'dfthcsos' / 'submission.npz')

r = CV(sub_dir / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
